In [49]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt 

In [50]:
df=pd.read_csv("housing.csv")

In [51]:
df['income_cat']=pd.cut(df['median_income'],bins=[0, 1.5, 3.0, 4.5, 6.0 , np.inf] , labels=[1,2,3,4,5])

In [52]:
from sklearn.model_selection import StratifiedShuffleSplit

split= StratifiedShuffleSplit(n_splits=1 , test_size=0.2 , random_state=42)


for train_index , test_index in split.split(df ,df['income_cat']):

    strat_train_set=df.loc[train_index]  ## its a original dataframe ie source

    strat_test_set=df.loc[test_index]     ##its a original dataframe ie source 

In [53]:
#let remove income_cat col

for sett in (strat_train_set , strat_test_set):
    
    sett.drop('income_cat' , axis=1, inplace=True)

In [54]:
df=strat_train_set.copy()

In [55]:
 df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16512 entries, 12655 to 19773
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           16512 non-null  float64
 1   latitude            16512 non-null  float64
 2   housing_median_age  16512 non-null  float64
 3   total_rooms         16512 non-null  float64
 4   total_bedrooms      16354 non-null  float64
 5   population          16512 non-null  float64
 6   households          16512 non-null  float64
 7   median_income       16512 non-null  float64
 8   median_house_value  16512 non-null  float64
 9   ocean_proximity     16512 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.4+ MB


In [56]:
housing=strat_train_set.drop('median_house_value' ,axis=1 ) ##features (did not write inplace=true not want change in original dataset ie strat_train)

housing_labels= strat_train_set['median_house_value'].copy()   ##labels

In [57]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")              ## imputer is now a robot will do everyting, here strategy is median , can be mean ....

# housing_num= housing.select_dtypes(include=[np.number])  ## want to add categorical data to give fix no.  ie - ocean proximity , so i add  comment 
                                                              

imputer.fit(housing_num)                                 ##now robot is analyze all table value 

SimpleImputer(strategy='median')

In [58]:
x=imputer.transform(housing_num)                         ## now robot fill nan with mdeian nd made new table x  and  convert into np array 

In [59]:
x


array([[-1.2146e+02,  3.8520e+01,  2.9000e+01, ...,  2.2370e+03,
         7.0600e+02,  2.1736e+00],
       [-1.1723e+02,  3.3090e+01,  7.0000e+00, ...,  2.0150e+03,
         7.6800e+02,  6.3373e+00],
       [-1.1904e+02,  3.5370e+01,  4.4000e+01, ...,  6.6700e+02,
         3.0000e+02,  2.8750e+00],
       ...,
       [-1.2272e+02,  3.8440e+01,  4.8000e+01, ...,  4.5800e+02,
         1.7200e+02,  3.1797e+00],
       [-1.2270e+02,  3.8310e+01,  1.4000e+01, ...,  1.2080e+03,
         5.0100e+02,  4.1964e+00],
       [-1.2214e+02,  3.9970e+01,  2.7000e+01, ...,  6.2500e+02,
         1.9700e+02,  3.1319e+00]])

In [60]:
housing=pd.DataFrame(x ,columns=housing_num.columns , index=housing_num.index)  # convert np arr ie x again to housing  dataframe

In [61]:
housing['ocean_proximity']= df['ocean_proximity']           #add caterogical data again in housing dataframe , as NAN is filled now with median 


In [62]:
housing = housing[['ocean_proximity']]  #this is done for one  hot encoder we only include category col as it divide into subpart - inland , bay ...


In [63]:
housing      

,ocean_proximity
12655,INLAND
15502,NEAR OCEAN
2908,INLAND
14053,NEAR OCEAN
20496,<1H OCEAN
...,...
15174,<1H OCEAN
12661,INLAND
19263,<1H OCEAN
19140,<1H OCEAN


In [64]:
set(housing['ocean_proximity'])

{'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}

In [65]:
# ORDINAL ENCODING IS USED WHEN THERE IS ORDER MEANS EACH DATA IS RALTED WE GO FAR POWER REDUCES LIKE - ONLINE RATING so model will learn 

#   that pattrern like replacing value in inc order - 1H OCEAN -1 ,  INLAND-2 ,  ISLAND-3 , NEAR BAY-4 , NEAR OCEAN-5


In [66]:
# from sklearn.preprocessing import OrdinalEncoder

In [67]:
# ordinal_encoder= OrdinalEncoder()

# housing_cat=  ordinal_encoder.fit_transform(housing)   ## convert into numpy array 

In [68]:
# housing_cat = pd.DataFrame(housing_cat , columns= housing.columns ,index=housing.index)

In [69]:
# ONE HOT ENCODER FOR UNORDER 

In [70]:
from sklearn.preprocessing import OneHotEncoder

In [71]:
hot_encoder=OneHotEncoder()

In [72]:
housing_cat=hot_encoder.fit_transform(housing)

In [73]:
housing_cat.toarray()

array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 1., 0., 0., 0.],
       ...,
       [1., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0.]])

In [74]:
hot_encoder.categories_

[array(['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'],
       dtype=object)]

In [75]:
housing_cat=pd.DataFrame(housing_cat.toarray() , columns=['<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'] ,index=housing.index)

In [76]:
housing_cat

,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,0.0,1.0,0.0,0.0,0.0
15502,0.0,0.0,0.0,0.0,1.0
2908,0.0,1.0,0.0,0.0,0.0
14053,0.0,0.0,0.0,0.0,1.0
20496,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...
15174,1.0,0.0,0.0,0.0,0.0
12661,0.0,1.0,0.0,0.0,0.0
19263,1.0,0.0,0.0,0.0,0.0
19140,1.0,0.0,0.0,0.0,0.0


In [79]:
df=pd.concat([df , housing_cat] , axis=1)

In [83]:
df=df.drop('ocean_proximity' , axis=1)

In [84]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736,72100.0,0.0,1.0,0.0,0.0,0.0
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373,279600.0,0.0,0.0,0.0,0.0,1.0
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750,82700.0,0.0,1.0,0.0,0.0,0.0
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264,112500.0,0.0,0.0,0.0,0.0,1.0
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964,238300.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900,268500.0,1.0,0.0,0.0,0.0,0.0
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139,90400.0,0.0,1.0,0.0,0.0,0.0
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797,140400.0,1.0,0.0,0.0,0.0,0.0
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964,258100.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
# MIN MAX SCALING KNOWN AS -   NORMALIZATION 



# scaling is done when is is huge difference between value eg - rating 1- 10 same as rating 1- 100...

# use this scaler when there is   * no outlier *    eg - weight of person 64, 75 suddenly 1000 this is outlier data entered by mistake 

In [86]:
# from sklearn.preprocessing import MinMaxScaler

In [87]:
# scaler= MinMaxScaler(feature_range=(-1,1))

In [89]:
# df_scaled= scaler.fit_transform(df)

In [91]:
# df_scaled = pd.DataFrame( df_scaled , columns = df.columns , index= df.index)

In [95]:
# df_scaled

In [94]:




# STANDARD SCALING KNOWN - STANDARIZATION 

# WHEN THERE IS UTLIERS USE THIS STANDARD SCALER



In [96]:
from sklearn.preprocessing import StandardScaler

In [97]:
scaler = StandardScaler()

In [98]:
df_scaled= scaler.fit_transform(df)

In [100]:
df_scaled = pd.DataFrame(df_scaled , columns = df.columns  ,  index=  df.index)

In [101]:
df_scaled

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
12655,-0.941350,1.347438,0.027564,0.584777,0.635123,0.732602,0.556286,-0.893647,-1.166015,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
15502,1.171782,-1.192440,-1.722018,1.261467,0.775677,0.533612,0.721318,1.292168,0.627451,-0.887683,-0.68391,-0.011006,-0.354889,2.602693
2908,0.267581,-0.125972,1.220460,-0.469773,-0.545045,-0.674675,-0.524407,-0.525434,-1.074397,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
14053,1.221738,-1.351474,-0.370069,-0.348652,-0.038567,-0.467617,-0.037297,-0.865929,-0.816829,-0.887683,-0.68391,-0.011006,-0.354889,2.602693
20496,0.437431,-0.635818,-0.131489,0.427179,0.269198,0.374060,0.220898,0.325752,0.270486,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15174,1.251711,-1.220505,-1.165333,1.890456,1.686854,0.543471,1.341519,0.637374,0.531511,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
12661,-0.921368,1.342761,-1.085806,2.468471,2.149712,3.002174,2.451492,-0.557509,-1.007844,-0.887683,1.46218,-0.011006,-0.354889,-0.384217
19263,-1.570794,1.310018,1.538566,-0.895802,-0.894007,-0.862013,-0.865118,-0.365475,-0.575684,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
19140,-1.560803,1.249211,-1.165333,0.249005,0.109257,-0.189747,0.010616,0.168261,0.441622,1.126529,-0.68391,-0.011006,-0.354889,-0.384217
